<a href="https://colab.research.google.com/github/kevin-mumaw/Options-Strategy-Analyzer/blob/main/options_scanner_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Options Scanner v4 — Phase 1

**CALL-side scanner with quality scoring gate**

A signal is only generated when conviction score >= 6/10.
If nothing qualifies, the scanner tells you to wait.
Every signal includes exact entry, stop, target, and time stop.

---

| Score | Conviction | Action |
|-------|------------|--------|
| 9-10  | VERY HIGH  | Strong candidate |
| 7-8   | HIGH       | Good candidate |
| 6     | MODERATE   | Minimum threshold — smaller size |
| < 6   | NONE       | No signal — wait |

**Scoring breakdown (10 pts total):**
Regime (2) + RSI (2) + Trend (2) + Volume (2) + Weekly (1) + Support (1)

In [12]:
import options_scanner_v4 as s
print("target_dte:", s.CONFIG["target_dte"])
print("min_reward_risk:", s.CONFIG["min_reward_risk"])

target_dte: 52
min_reward_risk: 2.0


In [13]:
# ─────────────────────────────────────────────────────────────
# FORCE RELOAD CELL — run this whenever options_scanner_v4.py
# is updated on GitHub. DO NOT DELETE.
# ─────────────────────────────────────────────────────────────
import sys, os, time

if 'options_scanner_v4' in sys.modules:
    del sys.modules['options_scanner_v4']
if os.path.exists('options_scanner_v4.py'):
    os.remove('options_scanner_v4.py')

url = f"https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py?t={int(time.time())}"
!wget -q -O options_scanner_v4.py "{url}"

# Verify correct version loaded
with open('options_scanner_v4.py') as f:
    content = f.read()

checks = ['target_width', 'min_reward_risk', '2.0', 'also_qualified', 'hard_fail']
for c in checks:
    print(f"{'✓' if c in content else '✗'} {c}")

from options_scanner_v4 import run_scan, print_results, deep_dive, ALL_SYMBOLS, CONFIG, find_best_call
print('\n✓ Scanner reloaded successfully')

✓ target_width
✓ min_reward_risk
✓ 2.0
✓ also_qualified
✓ hard_fail

✓ Scanner reloaded successfully


In [2]:
import yfinance as yf

for sym in ["C", "JPM", "V"]:
    ticker = yf.Ticker(sym)
    price = ticker.history(period="5d")["Close"].iloc[-1]
    print(f"\n{sym} — price: ${price:.2f}")

    # Show available expirations
    print(f"  Expirations: {ticker.options[:5]}")

    # Show options near ATM for July 17
    try:
        chain = ticker.option_chain("2026-07-17")
        calls = chain.calls
        atm = calls[
            (calls["strike"] >= price * 0.90) &
            (calls["strike"] <= price * 1.05)
        ][["strike","bid","ask","volume","openInterest"]].head(8)
        print(atm.to_string(index=False))
    except Exception as e:
        print(f"  Error: {e}")


C — price: $125.09
  Expirations: ('2026-05-29', '2026-06-05', '2026-06-12', '2026-06-18', '2026-06-26')
 strike   bid   ask  volume  openInterest
  115.0 12.40 13.90       1          1066
  120.0  9.15 10.15      20          4230
  125.0  6.50  6.80     171          3277
  130.0  4.00  4.45     433          4495

JPM — price: $306.38
  Expirations: ('2026-05-29', '2026-06-05', '2026-06-12', '2026-06-18', '2026-06-26')
 strike   bid   ask  volume  openInterest
  280.0 30.05 31.50     1.0           188
  285.0 25.55 27.45    13.0           353
  290.0 22.00 23.55     9.0           674
  295.0 18.40 19.95     2.0           823
  300.0 16.10 16.65   124.0          1236
  305.0 13.00 13.75   263.0          1149
  310.0 10.60 11.30   341.0          1894
  315.0  8.25  9.15   153.0          2156

V — price: $328.88
  Expirations: ('2026-05-29', '2026-06-05', '2026-06-12', '2026-06-18', '2026-06-26')
 strike   bid   ask  volume  openInterest
  300.0 32.50 34.70     1.0           170
  305.0 

In [3]:
import yfinance as yf

for sym in ["C", "JPM", "V"]:
    ticker = yf.Ticker(sym)
    print(f"\n{sym} — all expirations:")
    print(ticker.options)


C — all expirations:
('2026-05-29', '2026-06-05', '2026-06-12', '2026-06-18', '2026-06-26', '2026-07-02', '2026-07-17', '2026-08-21', '2026-09-18', '2026-10-16', '2026-11-20', '2026-12-18', '2027-01-15', '2027-03-19', '2027-06-17', '2028-01-21', '2028-12-15')

JPM — all expirations:
('2026-05-29', '2026-06-05', '2026-06-12', '2026-06-18', '2026-06-26', '2026-07-02', '2026-07-17', '2026-08-21', '2026-09-18', '2026-10-16', '2026-11-20', '2026-12-18', '2027-01-15', '2027-03-19', '2027-06-17', '2027-12-17', '2028-01-21', '2028-12-15')

V — all expirations:
('2026-05-29', '2026-06-05', '2026-06-12', '2026-06-18', '2026-06-26', '2026-07-02', '2026-07-17', '2026-08-21', '2026-09-18', '2026-11-20', '2026-12-18', '2027-01-15', '2027-03-19', '2027-06-17', '2027-12-17', '2028-01-21', '2028-12-15')


In [14]:
from options_scanner_v4 import black_scholes_greeks, safe_float, CONFIG
import yfinance as yf
from datetime import datetime
import numpy as np

sym = "JPM"
ticker = yf.Ticker(sym)
price = float(ticker.history(period="5d")["Close"].iloc[-1])
print(f"JPM price: ${price:.2f}")

# Replicate exactly what find_best_call does
today = datetime.today().date()
expirations = ticker.options
best_exp = None
best_dte = None
for exp in expirations:
    exp_date = datetime.strptime(exp, "%Y-%m-%d").date()
    dte = (exp_date - today).days
    if dte >= CONFIG["min_dte"]:
        if best_dte is None or abs(dte - CONFIG["target_dte"]) < abs(best_dte - CONFIG["target_dte"]):
            best_exp = exp
            best_dte = dte
        if dte > 75:
            break

print(f"Selected expiry: {best_exp} ({best_dte} DTE)")

chain = ticker.option_chain(best_exp)
calls = chain.calls.copy()
calls = calls[(calls["bid"] > 0) & (calls["ask"] > calls["bid"]) & (calls["lastPrice"] > 0)].copy()
calls = calls[(calls["volume"].fillna(0) >= 50) | (calls["openInterest"].fillna(0) >= 200)].copy()
calls["dist"] = abs(calls["strike"] - price) / price
calls = calls[calls["dist"] < CONFIG["atm_tolerance"]].sort_values("dist")

print(f"Candidates after all filters: {len(calls)}")
print("\nChecking each strike:")
for _, row in calls.head(8).iterrows():
    ask = safe_float(row["ask"])
    bid = safe_float(row["bid"])
    spread_pct = (ask - bid) / ask * 100
    iv = safe_float(row.get("impliedVolatility", 0), 0)
    T = best_dte / 365.0
    greeks = black_scholes_greeks("call", price, float(row["strike"]), T, 0.045, iv if iv > 0 else 0.25)
    delta = greeks.get("delta", 0.5)
    spread_ok = spread_pct <= CONFIG["max_spread_pct"]
    delta_ok = not np.isnan(delta) and 0.40 <= delta <= 0.65
    print(f"  ${row['strike']:.0f}  spread {spread_pct:.1f}% {'✓' if spread_ok else '✗'}  delta {delta:.3f} {'✓' if delta_ok else '✗'}")

JPM price: $306.38
Selected expiry: 2026-07-17 (53 DTE)
Candidates after all filters: 14

Checking each strike:
  $305  spread 5.5% ✓  delta 0.562 ✓
  $310  spread 6.2% ✓  delta 0.501 ✓
  $300  spread 3.3% ✓  delta 0.621 ✓
  $315  spread 9.8% ✓  delta 0.440 ✓
  $295  spread 7.8% ✓  delta 0.675 ✗
  $320  spread 12.1% ✓  delta 0.375 ✗
  $290  spread 6.6% ✓  delta 0.723 ✗
  $325  spread 14.0% ✓  delta 0.314 ✗


In [4]:
from options_scanner_v4 import find_best_call, CONFIG
import yfinance as yf

for sym in ["C", "JPM", "V"]:
    ticker = yf.Ticker(sym)
    price = ticker.history(period="5d")["Close"].iloc[-1]
    print(f"\n{sym} at ${price:.2f}")

    result = find_best_call(ticker, price, CONFIG)
    if result:
        print(f"  Found: ${result['strike']} call, delta {result['delta']}")
    else:
        print(f"  ✗ No option returned")

        # Diagnostic: check what's in the chain
        chain = ticker.option_chain("2026-07-17")
        calls = chain.calls

        # Apply each filter step by step
        q = calls[(calls["bid"] > 0) & (calls["ask"] > calls["bid"]) & (calls["lastPrice"] > 0)]
        print(f"  After quality filter: {len(q)} options")

        l = q[(q["volume"].fillna(0) >= 50) | (q["openInterest"].fillna(0) >= 200)]
        print(f"  After liquidity filter: {len(l)} options")

        l = l.copy()
        l["dist"] = abs(l["strike"] - price) / price
        a = l[l["dist"] < 0.15]
        print(f"  Within 15% ATM: {len(a)} options")

        if not a.empty:
            print(f"  Strikes available: {sorted(a['strike'].tolist())}")


C at $125.09
  Found: $125.0 call, delta 0.548

JPM at $306.38
  Found: $305.0 call, delta 0.562

V at $328.88
  Found: $330.0 call, delta 0.532


In [5]:
from options_scanner_v4 import black_scholes_greeks
import yfinance as yf
from datetime import datetime

for sym, min_d, max_d in [("C", 0.50, 0.70), ("JPM", 0.40, 0.65), ("V", 0.40, 0.65)]:
    ticker = yf.Ticker(sym)
    price = ticker.history(period="5d")["Close"].iloc[-1]
    chain = ticker.option_chain("2026-07-17")
    calls = chain.calls.copy()
    calls["dist"] = abs(calls["strike"] - price) / price
    atm = calls[calls["dist"] < 0.15].sort_values("dist")

    print(f"\n{sym} at ${price:.2f} — delta range {min_d}-{max_d}")
    for _, row in atm.head(6).iterrows():
        bid = float(row["bid"]) if row["bid"] > 0 else 0
        ask = float(row["ask"])
        spread_pct = (ask - bid) / ask * 100 if ask > 0 else 99
        iv = float(row.get("impliedVolatility", 0.25)) or 0.25
        T = (datetime.strptime("2026-07-17", "%Y-%m-%d").date() - datetime.today().date()).days / 365
        g = black_scholes_greeks("call", price, float(row["strike"]), T, 0.045, iv)
        delta = g["delta"]
        passed = "✓" if (not __import__('numpy').isnan(delta) and min_d <= delta <= max_d) else "✗"
        print(f"  {passed} ${row['strike']:.0f}  spread {spread_pct:.1f}%  delta {delta:.3f}  iv {iv*100:.1f}%")


C at $125.09 — delta range 0.5-0.7
  ✓ $125  spread 4.4%  delta 0.548  iv 35.2%
  ✗ $130  spread 10.1%  delta 0.427  iv 33.8%
  ✓ $120  spread 9.9%  delta 0.655  iv 39.0%
  ✗ $135  spread 11.7%  delta 0.313  iv 33.2%
  ✗ $115  spread 10.8%  delta 0.739  iv 42.7%
  ✗ $140  spread 14.7%  delta 0.216  iv 32.8%

JPM at $306.38 — delta range 0.4-0.65
  ✓ $305  spread 5.5%  delta 0.562  iv 27.8%
  ✓ $310  spread 6.2%  delta 0.501  iv 27.6%
  ✓ $300  spread 3.3%  delta 0.621  iv 28.4%
  ✓ $315  spread 9.8%  delta 0.440  iv 27.3%
  ✗ $295  spread 7.8%  delta 0.675  iv 29.3%
  ✗ $320  spread 12.1%  delta 0.375  iv 26.4%

V at $328.88 — delta range 0.4-0.65
  ✓ $330  spread 5.1%  delta 0.532  iv 24.3%
  ✓ $325  spread 3.8%  delta 0.595  iv 25.0%
  ✓ $335  spread 5.4%  delta 0.466  iv 23.8%
  ✓ $320  spread 7.1%  delta 0.648  iv 27.0%
  ✗ $340  spread 4.2%  delta 0.399  iv 23.3%
  ✗ $315  spread 8.3%  delta 0.693  iv 29.1%


In [6]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy'])
print('Dependencies ready')

Dependencies ready


In [7]:
# Cell 2 — Load scanner from GitHub
# Replace YOUR_USERNAME with kevin-mumaw
!wget -q https://raw.githubusercontent.com/kevin-mumaw/options-strategy-analyzer/main/options_scanner_v4.py

from options_scanner_v4 import (
    run_scan, print_results, deep_dive,
    ALL_SYMBOLS, WATCHLIST, CONFIG
)
print('Scanner loaded')
print(f'Watchlist: {len(ALL_SYMBOLS)} symbols')
print('Groups:', list(WATCHLIST.keys()))

Scanner loaded
Watchlist: 34 symbols
Groups: ['tier1_affordable', 'tier2_sweet_spot', 'tier3_mid_price', 'tier4_high_price']


In [8]:
# Cell 3 — Configure
# Edit account size here. Everything else auto-adjusts.

MY_CONFIG = {**CONFIG}  # copy defaults

MY_CONFIG['account_size'] = 5000.00  # update as your account grows

# Optional: scan only specific symbols instead of full watchlist
# MY_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM']
MY_SYMBOLS = None  # None = scan all 25 symbols

print(f'Account: ${MY_CONFIG["account_size"]:,.0f}')
print(f'Min score to generate signal: {MY_CONFIG["min_score"]}/10')
print(f'Stop loss: -{int(MY_CONFIG["stop_loss_pct"]*100)}% | '
      f'Profit target: +{int(MY_CONFIG["profit_target_pct"]*100)}% | '
      f'Time stop: {MY_CONFIG["time_stop_dte"]} DTE')

Account: $5,000
Min score to generate signal: 6/10
Stop loss: -35% | Profit target: +75% | Time stop: 21 DTE


In [9]:
# Cell 4 — Run the scan
# This is the main cell. Run this daily.
results = run_scan(symbols=MY_SYMBOLS, config=MY_CONFIG)
print_results(results)


  Fetching market regime (QQQ)...
  Regime: BULLISH — QQQ 717.54 | MA50 642.01 (+11.8%) | MA200 613.6 (+16.9%)

  Scanning 34 symbols.................................. done.


██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — Phase 1  |  2026-05-25 00:29
  Account: $5,000  |  Scanned: 34 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 717.54 | MA50 642.01 (+11.8%) | MA200 613.6 (+16.9%)

  2 signal(s) found:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SIGNAL #1  |  BAC — BUY CALL  |  Score: 7/10  [HIGH]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  WHY THIS TRADE:
    ✓ Regime [2/2] BULLISH — QQQ above both MAs
    ~ RSI     [1/2] Neutral — RSI 48 (room to run)
    ✓ Trend   [2/2] Clean uptrend — price > MA20 > MA50
    ~ Weekly  [0/1] Weekly trend mixed
    ~ Volume  [1/2] Mild accumulation — 2/5 up days on above-avg volume
    ✓ Support [1/1] Near MA50 

In [10]:
# Cell 5 — Deep dive on a specific symbol
# Use this to investigate any symbol in detail,
# whether it appeared in the scan or not.

SYMBOL = 'bac'  # change this

deep_dive(SYMBOL, config=MY_CONFIG)


  Fetching regime and analyzing bac...

██████████████████████████████████████████████████████████████
  OPTIONS SCANNER — Phase 1  |  2026-05-25 00:29
  Account: $5,000  |  Scanned: 1 symbols
██████████████████████████████████████████████████████████████

  MARKET REGIME: 🟢 BULLISH
  QQQ 717.54 | MA50 642.01 (+11.8%) | MA200 613.6 (+16.9%)

  1 signal(s) found:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  SIGNAL #1  |  bac — BUY CALL  |  Score: 7/10  [HIGH]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  WHY THIS TRADE:
    ✓ Regime [2/2] BULLISH — QQQ above both MAs
    ~ RSI     [1/2] Neutral — RSI 48 (room to run)
    ✓ Trend   [2/2] Clean uptrend — price > MA20 > MA50
    ~ Weekly  [0/1] Weekly trend mixed
    ~ Volume  [1/2] Mild accumulation — 2/5 up days on above-avg volume
    ✓ Support [1/1] Near MA50 support (+1.9%)

  STOCK:
    Price: $51.80   RSI: 48   Trend: UPTREND
    MA20:  $51.73 (+0.1%)   MA50: $50.84 (+1.9%)
    HV30:  19.5% 

In [11]:
# Cell 6 — Review all scores from last scan
# See every symbol's score ranked highest to lowest.

import pandas as pd

rows = []
for r in results['all_results']:
    if not r.get('error'):
        rows.append({
            'Symbol':    r['symbol'],
            'Score':     r['score'],
            'Conviction':r['conviction'],
            'Trend':     r.get('trend','—'),
            'RSI':       r.get('rsi','—'),
            'Volume':    r.get('vol_data',{}).get('label','—'),
            'Weekly':    r.get('weekly',{}).get('trend','—'),
            'Has Option':r.get('option') is not None,
        })

df = pd.DataFrame(rows).sort_values('Score', ascending=False)
print(df.to_string(index=False))

Symbol  Score Conviction     Trend  RSI            Volume    Weekly  Has Option
  COST      8       HIGH   UPTREND 53.6           NEUTRAL   UPTREND       False
   BAC      7       HIGH   UPTREND 47.5 MILD_ACCUMULATION     MIXED        True
   XLF      7       HIGH   UPTREND 54.0           NEUTRAL     MIXED        True
     C      7       HIGH     MIXED 48.8           NEUTRAL   UPTREND       False
   JPM      7       HIGH   UPTREND 48.8           NEUTRAL     MIXED       False
     V      7       HIGH   UPTREND 52.9           NEUTRAL     MIXED       False
   XOM      7       HIGH     MIXED 53.0           NEUTRAL   UPTREND       False
   AAL      6   MODERATE   UPTREND 71.4      ACCUMULATION     MIXED       False
    KO      6   MODERATE   UPTREND 75.7           NEUTRAL   UPTREND       False
   DIS      6   MODERATE     MIXED 53.4           NEUTRAL     MIXED       False
   AMD      6   MODERATE   UPTREND 75.2           NEUTRAL   UPTREND       False
  INTC      6   MODERATE   UPTREND 66.2 